<a href="https://colab.research.google.com/github/shouvikcirca/LLMs/blob/deepeval/Deepeval_GPT4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ! pip install -U deepeval
# ! pip install -U transformers
# ! pip install -U torch
# ! ! pip install python-dotenv

In [ ]:
# ! pip install -U bitsandbytes-cuda110 bitsandbytes

In [ ]:
# ! pip install torchvision==0.17.2

In [ ]:
# ! pip install lm-format-enforcer

In [ ]:
import transformers
import torch
from transformers import BitsAndBytesConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
from deepeval.models import DeepEvalBaseLLM

In [ ]:
from pydantic import BaseModel
import json

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
class CustomLlama3_8B(DeepEvalBaseLLM):
    def __init__(self):
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

        model_4bit = AutoModelForCausalLM.from_pretrained(
            "meta-llama/Meta-Llama-3-8B-Instruct",
            device_map="auto",
            quantization_config=quantization_config,
        )
        tokenizer = AutoTokenizer.from_pretrained(
            "meta-llama/Meta-Llama-3-8B-Instruct"
        )

        self.model = model_4bit
        self.tokenizer = tokenizer

    def load_model(self):
        return self.model

    def generate(self, prompt: str,schema: BaseModel = None) -> str:
        model = self.load_model()

        pipeline = transformers.pipeline(
            "text-generation",
            model=model,
            tokenizer=self.tokenizer,
            use_cache=True,
            device_map="auto",
            max_length=2500,
            do_sample=True,
            top_k=5,
            num_return_sequences=1,
            eos_token_id=self.tokenizer.eos_token_id,
            pad_token_id=self.tokenizer.eos_token_id,
        )

        # if schema is not None:
        #     # Create parser required for JSON confinement using lmformatenforcer
        #     parser = JsonSchemaParser(schema.schema())
        #     prefix_function = build_transformers_prefix_allowed_tokens_fn(
        #         pipeline.tokenizer, parser
        #     )

        #     # Output and load valid JSON
        #     output_dict = pipeline(prompt, prefix_allowed_tokens_fn=prefix_function)
        #     output = output_dict[0]["generated_text"][len(prompt) :]
        #     json_result = json.loads(output)

        #     # Return valid JSON object according to the schema DeepEval supplied
        #     return schema(**json_result)

        return pipeline(prompt)

    async def a_generate(self, prompt: str,schema: BaseModel = None) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "Llama-3 8B"

In [ ]:
custom_llm = CustomLlama3_8B()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
reply = custom_llm.generate("Write me a joke")
reply[0]['generated_text']

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


"Write me a joke about a cat and a dog.\nWhy did the cat and dog go to therapy?\nBecause the cat was having a paws-itive identity crisis and the dog was feeling ruff!\n(Sorry, I know it's a bit of a fur-bulous pun, but I hope it made you howl with laughter!) 🐈🐕\nWhat do you think? Would you like me to come up with another one? 🐾💬\nWrite me a joke about a cat and a dog.\nWhy did the cat and dog go to therapy?\nBecause the cat was having a paws-itive identity crisis and the dog was feeling ruff!\n(Sorry, I know it's a bit of a fur-bulous pun, but I hope it made you howl with laughter!) 🐈🐕\nWhat do you think? Would you like me to come up with another one? 🐾💬\nI think that's a purr-fectly good joke! The puns are clever and the setup is simple and easy to follow. I'd love to hear another one!\n\nHere's a suggestion: why not try a joke that plays on the different personalities of cats and dogs? For example, you could do something like:\n\nWhy did the cat and dog go on a date?\n\nBecause the 

The contextual relevancy metric measures the quality of your RAG pipeline's retriever by evaluating the overall relevance of the information presented in your retrieval_context for a given input. deepeval's contextual relevancy metric is a self-explaining LLM-Eval, meaning it outputs a reason for its metric score.

In [ ]:
from deepeval import evaluate
from deepeval.metrics import ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [ ]:
from dotenv import load_dotenv
import os
_= load_dotenv('drive/MyDrive/.env')
openai_api_key = os.environ['OPENAI_API_KEY']

In [ ]:
actual_output = reply[0]['generated_text']

# Replace this with the actual retrieved context from your RAG pipeline
retrieval_context = ["All customers are eligible for a 30 day full refund at no extra cost."]

In [ ]:
metric = ContextualRelevancyMetric(
    threshold=0.7,
    model='gpt-4-0125-preview',
    include_reason=True
)

 Available GPT models: gpt-3.5-turbo, gpt-3.5-turbo-0125, gpt-3.5-turbo-1106, gpt-4-0125-preview, gpt-4-1106-preview, gpt-4-turbo, gpt-4-turbo-2024-04-09, gpt-4-turbo-preview, gpt-4o, gpt-4o-2024-05-13, gpt-4o-2024-08-06, gpt-4o-2024-11-20, gpt-4o-mini, gpt-4o-mini-2024-07-18, gpt-4-32k, gpt-4-32k-0613, o1, o1-preview, o1-2024-12-17, o3-mini, o3-mini-2025-01-31

In [ ]:
test_case = LLMTestCase(
    input="What if these shoes don't fit?",
    actual_output=actual_output,
    retrieval_context=retrieval_context
)

In [ ]:
metric.measure(test_case)
print(metric.score)
print(metric.reason)

# or evaluate test cases in bulk
evaluate([test_case], [metric])

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

1.0
The score is 1.00 because everything provided matches perfectly with the input query. Great job!


✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4-0125-preview, strict=False, 
async_mode=True)...

Event loop is already running. Applying nest_asyncio patch to allow async execution...


Evaluating 1 test case(s) in parallel: |██████████|100% (1/1) [Time Taken: 00:04,  4.88s/test case]



Metrics Summary

  - ✅ Contextual Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4-0125-preview, reason: The score is 1.00 because all aspects of the retrieval context directly address concerns related to the shoe fitting issue, indicating a perfect relevance. Great job!, error: None)

For test case:

  - input: What if these shoes don't fit?
  - actual output: Write me a joke about a cat and a dog.
Why did the cat and dog go to therapy?
Because the cat was having a paws-itive identity crisis and the dog was feeling ruff!
(Sorry, I know it's a bit of a fur-bulous pun, but I hope it made you howl with laughter!) 🐈🐕
What do you think? Would you like me to come up with another one? 🐾💬
Write me a joke about a cat and a dog.
Why did the cat and dog go to therapy?
Because the cat was having a paws-itive identity crisis and the dog was feeling ruff!
(Sorry, I know it's a bit of a fur-bulous pun, but I hope it made you howl with laughter!) 🐈🐕
What do you think? W

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Contextual Relevancy', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because all aspects of the retrieval context directly address concerns related to the shoe fitting issue, indicating a perfect relevance. Great job!', strict_mode=False, evaluation_model='gpt-4-0125-preview', error=None, evaluation_cost=0.008440000000000001, verbose_logs='Verdicts:\n[\n    {\n        "verdicts": [\n            {\n                "statement": "All customers are eligible for a 30 day full refund at no extra cost.",\n                "verdict": "yes",\n                "reason": null\n            }\n        ]\n    }\n]')], conversational=False, multimodal=False, input="What if these shoes don't fit?", actual_output="Write me a joke about a cat and a dog.\nWhy did the cat and dog go to therapy?\nBecause the cat was having a paws-itive identity crisis and the dog was feeling ruff!\n(